In [1]:
"""
Baseline model: predict civil_outcome for first_instance loan disputes.

Target: civil_outcome (6 classes: granted, granted_uncontested, settled,
withdrawn, partial, denied). law_articles and civil_award_amount are
deliberately excluded from features - both are extracted from the
post-decision operative section, so using them would leak the outcome
into its own predictor. case_narrative (free text) isn't used yet either;
that's a natural next step (NLP/LLM feature extraction), not part of
this first structured-features pass.

Run: python3 baseline_model.py
Needs: output.csv in the same folder (or edit CSV_PATH below).
"""
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

import features  # shared with predict_new_case.py - one definition, not two copies to keep in sync

CSV_PATH = "output.csv"  # <- set if not in the same folder as this script

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", None)


# ---------- load + filter to the labeled training set ----------
df = pd.read_csv(CSV_PATH, encoding="utf-8-sig")
fi = df[(df.case_level == "first_instance") & (df.civil_outcome.notna())].copy()
print(f"training rows: {len(fi)}")
print(fi.civil_outcome.value_counts())
print()


# ---------- feature engineering (see features.py for the actual logic) ----------
X, median_claim, judge_counts = features.build_training_features(fi)
y = fi["civil_outcome"]
labels = sorted(y.unique())

# TF-IDF on the raw narrative text, in ADDITION to the regex flags above -
# the flags target specific known-relevant concepts precisely, TF-IDF
# catches broader patterns no one would think to hand-code. Tested
# max_features from 150-2500: climbs steadily to 1000, then starts
# giving it back (overfitting past that point) - 1000 is the real
# sweet spot, not a round-number default.
preprocess = ColumnTransformer(
    [
        ("cat", OneHotEncoder(handle_unknown="ignore"), features.FEATURES_CAT),
        ("text", TfidfVectorizer(max_features=1000, min_df=5, ngram_range=(1, 2), sublinear_tf=True), features.TEXT_COL),
    ],
    remainder="passthrough",
)

# denied=8, partial=19 - stratified k-fold so every fold sees some of
# each class, but treat per-class numbers for these two as noisy no
# matter what; there just isn't much data to evaluate them on yet.
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


# ---------- naive baseline: always predict the majority class ----------
# This is the number a real model has to beat. Accuracy alone makes this
# look deceptively decent - that's exactly why macro-F1 is the metric
# used everywhere below instead.
majority_preds = pd.Series([y.mode()[0]] * len(y))
print("### naive baseline: always predict majority class ###")
print("accuracy:", round((majority_preds.values == y.values).mean(), 3))
print("macro-F1:", round(f1_score(y, majority_preds, average="macro"), 3))
print()


# ---------- logistic regression ----------
# with_mean=False since ColumnTransformer's one-hot output can be
# sparse - centering would break that. Scale-only still fixes the
# convergence warning, which was from mixing very different feature
# ranges (narrative_length_log spans ~0-11, judge_case_count up to 40+,
# the new boolean flags are 0/1) - exactly what the warning pointed at.
# Random forest doesn't need this, it's scale-invariant by construction.
lr = LogisticRegression(max_iter=2000, class_weight="balanced")
pipe_lr = Pipeline([("prep", preprocess), ("scale", StandardScaler(with_mean=False)), ("clf", lr)])
preds_lr = cross_val_predict(pipe_lr, X, y, cv=cv)
print("### logistic regression ###")
print("macro-F1:", round(f1_score(y, preds_lr, average="macro"), 3))
print()


# ---------- random forest ----------
rf = RandomForestClassifier(
    n_estimators=300, class_weight="balanced", random_state=42, min_samples_leaf=3
)
pipe_rf = Pipeline([("prep", preprocess), ("clf", rf)])
preds_rf = cross_val_predict(pipe_rf, X, y, cv=cv)
print("### random forest ###")
print(classification_report(y, preds_rf, zero_division=0))
print("macro-F1:", round(f1_score(y, preds_rf, average="macro"), 3))
print()
print("confusion matrix (rows=true, cols=predicted):")
cm = pd.DataFrame(confusion_matrix(y, preds_rf, labels=labels), index=labels, columns=labels)
print(cm.to_string())
print()


# ---------- example predictions on genuinely held-out rows ----------
# cross_val_predict never lets the model see a row's own label while
# predicting it - each prediction comes from a model trained only on
# the OTHER folds. That's the rigorous stand-in for "a new case the
# model hasn't seen," short of an actual undecided case from the live
# site (which - if you want a real one, not a simulated one - would
# mean checking whether shuukh.mn exposes case status/pending listings
# at all; everything scraped so far has been fully decided rulings).
examples = fi[["source_file", "civil_outcome"]].copy()
examples["predicted"] = preds_rf
examples["correct"] = examples.civil_outcome == examples.predicted
print("### example held-out predictions (model never saw these rows' own labels) ###")
print(examples.sample(10, random_state=1).to_string(index=False))
print()


# ---------- feature importances ----------
pipe_rf.fit(X, y)
feat_names = pipe_rf.named_steps["prep"].get_feature_names_out()
importances = pipe_rf.named_steps["clf"].feature_importances_
imp = pd.Series(importances, index=feat_names).sort_values(ascending=False)
print("### feature importances (random forest) ###")
print(imp.to_string())

training rows: 5153
civil_outcome
granted                2885
settled                 828
granted_uncontested     483
partial                 460
withdrawn               377
denied                  120
Name: count, dtype: int64

### naive baseline: always predict majority class ###
accuracy: 0.56
macro-F1: 0.12

### logistic regression ###
macro-F1: 0.549

### random forest ###
                     precision    recall  f1-score   support

             denied       0.10      0.15      0.12       120
            granted       0.81      0.77      0.79      2885
granted_uncontested       0.85      0.54      0.66       483
            partial       0.48      0.76      0.59       460
            settled       0.81      0.74      0.77       828
          withdrawn       0.65      0.76      0.70       377

           accuracy                           0.73      5153
          macro avg       0.62      0.62      0.61      5153
       weighted avg       0.76      0.73      0.74      5153

macro-

In [6]:
"""
Shared feature engineering for civil_outcome prediction. Both
baseline_model.py (training/evaluation) and predict_new_case.py
(deployment) import from here - one definition, so the two can't
quietly drift out of sync with each other over time.
"""
import numpy as np
import pandas as pd

FEATURES_NUM = [
    "claim_amount_log", "claim_amount_missing", "judge_case_count",
    "narrative_missing", "narrative_length_log",
    "has_interest_rate", "has_institutional_lender", "has_app_lending",
    "has_collateral", "has_guarantor", "has_gov_fund",
]
FEATURES_CAT = ["plaintiff_type", "defendant_type", "court_type"]
TEXT_COL = "narrative_filled"
ALL_FEATURE_COLS = FEATURES_NUM + FEATURES_CAT + [TEXT_COL]


def infer_court_type(court):
    return "urban" if "дүүргийн" in str(court) else "rural"


def build_narrative_flags(narrative_filled):
    """narrative_filled: a pandas Series of strings (already .fillna(''))."""
    return pd.DataFrame({
        "has_interest_rate": narrative_filled.str.contains(r"хүү|хувь.{0,10}тэй", regex=True).astype(int),
        "has_institutional_lender": narrative_filled.str.contains(r"ХХК|ББСБ", regex=True).astype(int),
        "has_app_lending": narrative_filled.str.contains(r"аппликейшн|апп\b", regex=True).astype(int),
        "has_collateral": narrative_filled.str.contains(r"барьцаа", regex=True).astype(int),
        "has_guarantor": narrative_filled.str.contains(r"батлан", regex=True).astype(int),
        "has_gov_fund": narrative_filled.str.contains(r"хөгжүүлэх сан|Засаг дарга", regex=True).astype(int),
    })


def build_training_features(fi, median_claim=None, judge_counts=None):
    """
    fi: dataframe with raw columns (claim_amount_requested, judge, court,
    plaintiff_type, defendant_type, case_narrative). Returns (X, median_claim,
    judge_counts) - the last two are computed here if not passed in, and
    should be SAVED alongside the trained model so predict_new_case.py can
    reuse the exact same values instead of recomputing (and drifting) later.
    """
    fi = fi.copy()
    if median_claim is None:
        median_claim = fi.claim_amount_requested.median()
    if judge_counts is None:
        judge_counts = fi.judge.value_counts()

    fi["court_type"] = fi.court.apply(infer_court_type)
    fi["judge_case_count"] = fi.judge.map(judge_counts).fillna(judge_counts.median())
    fi["claim_amount_missing"] = fi.claim_amount_requested.isna().astype(int)
    fi["claim_amount_log"] = np.log1p(fi.claim_amount_requested.fillna(median_claim))
    fi["plaintiff_type"] = fi.plaintiff_type.fillna("unknown")
    fi["defendant_type"] = fi.defendant_type.fillna("unknown")
    fi["narrative_missing"] = fi.case_narrative.isna().astype(int)
    narrative = fi.case_narrative.fillna("")
    fi["narrative_length_log"] = np.log1p(narrative.str.len())
    fi[TEXT_COL] = narrative
    fi = pd.concat([fi, build_narrative_flags(narrative)], axis=1)

    return fi[ALL_FEATURE_COLS], median_claim, judge_counts


def build_case_features(claim_amount, plaintiff_type, defendant_type, court_type,
                         narrative_text, judge_name=None, median_claim=None, judge_counts=None):
    """
    For ONE new, real case. court_type must be 'urban' or 'rural' directly
    (caller decides, or pass a court name through infer_court_type first).
    judge_name: if the assigned judge is known and appears in judge_counts,
    uses their real historical count; otherwise falls back to the median
    judge_case_count across training data - same logic as the demo, now
    backed by the real model.
    """
    narrative_filled = pd.Series([narrative_text or ""])
    if judge_name is not None and judge_counts is not None and judge_name in judge_counts.index:
        jcc = judge_counts[judge_name]
    elif judge_counts is not None:
        jcc = judge_counts.median()
    else:
        jcc = 24.0  # fallback if no historical judge data was supplied at all

    row = pd.DataFrame({
        "claim_amount_log": [np.log1p(claim_amount if claim_amount is not None else (median_claim or 0))],
        "claim_amount_missing": [1 if claim_amount is None else 0],
        "judge_case_count": [jcc],
        "narrative_missing": [1 if not narrative_text else 0],
        "narrative_length_log": [np.log1p(len(narrative_text or ""))],
        "plaintiff_type": [plaintiff_type or "unknown"],
        "defendant_type": [defendant_type or "unknown"],
        "court_type": [court_type],
        TEXT_COL: [narrative_text or ""],
    })
    row = pd.concat([row, build_narrative_flags(narrative_filled)], axis=1)
    return row[ALL_FEATURE_COLS]


In [ ]:
"""
Baseline model: predict civil_outcome for first_instance loan disputes.

Target: civil_outcome (6 classes: granted, granted_uncontested, settled,
withdrawn, partial, denied). law_articles and civil_award_amount are
deliberately excluded from features - both are extracted from the
post-decision operative section, so using them would leak the outcome
into its own predictor. case_narrative (free text) isn't used yet either;
that's a natural next step (NLP/LLM feature extraction), not part of
this first structured-features pass.

Run: python3 baseline_model.py
Needs: output.csv in the same folder (or edit CSV_PATH below).
"""
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

import features  # shared with predict_new_case.py - one definition, not two copies to keep in sync

CSV_PATH = "output.csv"  # <- set if not in the same folder as this script

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", None)


# ---------- load + filter to the labeled training set ----------
df = pd.read_csv(CSV_PATH, encoding="utf-8-sig")
fi = df[(df.case_level == "first_instance") & (df.civil_outcome.notna())].copy()
print(f"training rows: {len(fi)}")
print(fi.civil_outcome.value_counts())
print()


# ---------- feature engineering (see features.py for the actual logic) ----------
X, median_claim, judge_counts = features.build_training_features(fi)
y = fi["civil_outcome"]
labels = sorted(y.unique())

# TF-IDF on the raw narrative text, in ADDITION to the regex flags above -
# the flags target specific known-relevant concepts precisely, TF-IDF
# catches broader patterns no one would think to hand-code. Tested
# max_features from 150-2500: climbs steadily to 1000, then starts
# giving it back (overfitting past that point) - 1000 is the real
# sweet spot, not a round-number default.
preprocess = ColumnTransformer(
    [
        ("cat", OneHotEncoder(handle_unknown="ignore"), features.FEATURES_CAT),
        ("text", TfidfVectorizer(max_features=1000, min_df=5, ngram_range=(1, 2), sublinear_tf=True), features.TEXT_COL),
    ],
    remainder="passthrough",
)

# denied=8, partial=19 - stratified k-fold so every fold sees some of
# each class, but treat per-class numbers for these two as noisy no
# matter what; there just isn't much data to evaluate them on yet.
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


# ---------- naive baseline: always predict the majority class ----------
# This is the number a real model has to beat. Accuracy alone makes this
# look deceptively decent - that's exactly why macro-F1 is the metric
# used everywhere below instead.
majority_preds = pd.Series([y.mode()[0]] * len(y))
print("### naive baseline: always predict majority class ###")
print("accuracy:", round((majority_preds.values == y.values).mean(), 3))
print("macro-F1:", round(f1_score(y, majority_preds, average="macro"), 3))
print()


# ---------- logistic regression ----------
# with_mean=False since ColumnTransformer's one-hot output can be
# sparse - centering would break that. Scale-only still fixes the
# convergence warning, which was from mixing very different feature
# ranges (narrative_length_log spans ~0-11, judge_case_count up to 40+,
# the new boolean flags are 0/1) - exactly what the warning pointed at.
# Random forest doesn't need this, it's scale-invariant by construction.
lr = LogisticRegression(max_iter=2000, class_weight="balanced")
pipe_lr = Pipeline([("prep", preprocess), ("scale", StandardScaler(with_mean=False)), ("clf", lr)])
preds_lr = cross_val_predict(pipe_lr, X, y, cv=cv)
print("### logistic regression ###")
print("macro-F1:", round(f1_score(y, preds_lr, average="macro"), 3))
print()


# ---------- random forest ----------
rf = RandomForestClassifier(
    n_estimators=300, class_weight="balanced", random_state=42, min_samples_leaf=3
)
pipe_rf = Pipeline([("prep", preprocess), ("clf", rf)])
preds_rf = cross_val_predict(pipe_rf, X, y, cv=cv)
print("### random forest ###")
print(classification_report(y, preds_rf, zero_division=0))
print("macro-F1:", round(f1_score(y, preds_rf, average="macro"), 3))
print()
print("confusion matrix (rows=true, cols=predicted):")
cm = pd.DataFrame(confusion_matrix(y, preds_rf, labels=labels), index=labels, columns=labels)
print(cm.to_string())
print()


# ---------- example predictions on genuinely held-out rows ----------
# cross_val_predict never lets the model see a row's own label while
# predicting it - each prediction comes from a model trained only on
# the OTHER folds. That's the rigorous stand-in for "a new case the
# model hasn't seen," short of an actual undecided case from the live
# site (which - if you want a real one, not a simulated one - would
# mean checking whether shuukh.mn exposes case status/pending listings
# at all; everything scraped so far has been fully decided rulings).
examples = fi[["source_file", "civil_outcome"]].copy()
examples["predicted"] = preds_rf
examples["correct"] = examples.civil_outcome == examples.predicted
print("### example held-out predictions (model never saw these rows' own labels) ###")
print(examples.sample(10, random_state=1).to_string(index=False))
print()


# ---------- feature importances ----------
pipe_rf.fit(X, y)
feat_names = pipe_rf.named_steps["prep"].get_feature_names_out()
importances = pipe_rf.named_steps["clf"].feature_importances_
imp = pd.Series(importances, index=feat_names).sort_values(ascending=False)
print("### feature importances (random forest) ###")
print(imp.to_string())


training rows: 5153
civil_outcome
granted                2885
settled                 828
granted_uncontested     483
partial                 460
withdrawn               377
denied                  120
Name: count, dtype: int64

### naive baseline: always predict majority class ###
accuracy: 0.56
macro-F1: 0.12

### logistic regression ###
macro-F1: 0.549



In [13]:
"""
Train the real model (same pipeline as baseline_model.py: RandomForest
+ TF-IDF + regex narrative flags) on the FULL labeled dataset and save
it to disk - so predicting a new case doesn't mean retraining from
scratch every time.

Run: python3 train_and_save_model.py
Needs: output.csv and features.py in the same folder.
Produces: trained_model.joblib
"""
import joblib
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

import features

CSV_PATH = "output.csv"

df = pd.read_csv(CSV_PATH, encoding="utf-8-sig")
fi = df[(df.case_level == "first_instance") & (df.civil_outcome.notna())].copy()
print(f"training on {len(fi)} labeled first_instance rows")

X, median_claim, judge_counts = features.build_training_features(fi)
y = fi["civil_outcome"]

preprocess = ColumnTransformer(
    [
        ("cat", OneHotEncoder(handle_unknown="ignore"), features.FEATURES_CAT),
        ("text", TfidfVectorizer(max_features=1000, min_df=5, ngram_range=(1, 2)), features.TEXT_COL),
    ],
    remainder="passthrough",
)
rf = RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=42, min_samples_leaf=3)
pipe = Pipeline([("prep", preprocess), ("clf", rf)])
pipe.fit(X, y)

# save the fitted pipeline PLUS the lookups needed to featurize a brand
# new case the same way (median claim amount for missing values, and
# each judge's historical case count) - without these, predict_new_case.py
# would have no way to reproduce training-time feature values exactly.
joblib.dump(
    {"pipeline": pipe, "median_claim": median_claim, "judge_counts": judge_counts, "classes": list(pipe.classes_)},
    "trained_model.joblib",
)
print("saved -> trained_model.joblib")
print(f"classes: {list(pipe.classes_)}")

training on 4309 labeled first_instance rows
saved -> trained_model.joblib
classes: ['denied', 'granted', 'granted_uncontested', 'partial', 'settled', 'withdrawn']


In [1]:
"""
Predict civil_outcome for real, new (undecided) first-instance loan cases
using the actual trained model.

BATCH MODE (default):
    python3 predict_new_case.py [INPUT_FOLDER]

    Reads every .txt file in INPUT_FOLDER (default: ./new_cases/), parses
    each with shuukh_parser_v2, extracts the features the model needs,
    and prints ranked outcome probabilities per file. Also saves a summary
    CSV to predictions_output.csv.

SINGLE-CASE API:
    from predict_new_case import predict_case
    results = predict_case(claim_amount=..., plaintiff_type=..., ...)

Needs: trained_model.joblib in the same folder (run train_and_save_model.py
first if it doesn't exist yet).
"""
import csv
import sys
from pathlib import Path

import joblib

import features
import shuukh_parser_v2 as parser

MODEL_PATH = "trained_model.joblib"
INPUT_FOLDER = "new_cases"
OUTPUT_CSV = "predictions_output.csv"


def load_model():
    saved = joblib.load(MODEL_PATH)
    return saved["pipeline"], saved["median_claim"], saved["judge_counts"]


def predict_case(claim_amount, plaintiff_type, defendant_type, court_type,
                  narrative_text, judge_name=None, model=None):
    """
    claim_amount: number (₮), or None if unknown
    plaintiff_type / defendant_type: 'individual' or 'company'
    court_type: 'urban' (Ulaanbaatar district court) or 'rural' (aimag/province)
    narrative_text: the actual petition/claim text, in Mongolian - this is
        the single most informative input (TF-IDF over this text is now
        the model's top feature group), don't skip it if it's available
    judge_name: pass this if the judge is already assigned and known -
        otherwise the model uses the median caseload across training data
    """
    if model is None:
        pipe, median_claim, judge_counts = load_model()
    else:
        pipe, median_claim, judge_counts = model

    row = features.build_case_features(
        claim_amount=claim_amount,
        plaintiff_type=plaintiff_type,
        defendant_type=defendant_type,
        court_type=court_type,
        narrative_text=narrative_text,
        judge_name=judge_name,
        median_claim=median_claim,
        judge_counts=judge_counts,
    )
    proba = pipe.predict_proba(row)[0]
    classes = pipe.classes_
    ranked = sorted(zip(classes, proba), key=lambda x: -x[1])
    return ranked  # [(outcome, probability), ...] sorted highest first


def parse_and_predict_folder(input_folder, model=None):
    """
    Scan input_folder for .txt files, parse each with shuukh_parser_v2,
    extract the features the trained model needs, and return predictions.

    Returns a list of dicts, one per file, with parsed metadata + predictions.
    """
    folder = Path(input_folder)
    if not folder.is_dir():
        print(f"ERROR: Input folder '{folder}' does not exist.")
        print(f"Create it and drop .txt case files there, then re-run.")
        sys.exit(1)

    txt_files = sorted(
        p for p in folder.glob("*")
        if p.is_file() and not p.name.startswith(".") and p.name != "README.md" and p.name != OUTPUT_CSV
    )
    if not txt_files:
        print(f"No valid case files found in '{folder}/'.")
        print(f"Drop case narrative files there and re-run.")
        sys.exit(0)

    if model is None:
        model = load_model()

    results = []
    for filepath in txt_files:
        parsed = parser.parse_file(filepath)
        dispute_type = parsed.get("dispute_type") or ""
        case_level = parsed.get("case_level")

        # Refuse to predict on anything we can't confirm is in scope,
        # rather than silently giving a confident-looking prediction on
        # something the model was never trained on. This model was only
        # trained on first_instance "Бусад зээл" cases specifically -
        # note that's non-bank lending (ББСБ, app-based, installment,
        # gov-fund, individual loans) - "Банкны зээл" (bank loans) is a
        # SEPARATE category on the source site and was never in scope.
        skip_reason = None
        if not dispute_type:
            skip_reason = "dispute_type could not be extracted - can't confirm this is even a loan dispute"
        elif "Бусад зээл" not in dispute_type:
            skip_reason = f"dispute_type is '{dispute_type}', not 'Бусад зээл' - out of scope for this model (e.g. a bank loan, lease, or other dispute type wasn't part of training)"
        elif case_level != "first_instance":
            skip_reason = f"case_level is '{case_level}', not 'first_instance' - this model only predicts first-instance outcomes"

        if skip_reason:
            results.append({
                "source_file": filepath.name,
                "case_id": parsed.get("case_id"),
                "case_level": case_level,
                "dispute_type": parsed.get("dispute_type"),
                "court": parsed.get("court"),
                "judge": parsed.get("judge"),
                "plaintiff_type": parsed.get("plaintiff_type"),
                "defendant_type": parsed.get("defendant_type"),
                "court_type": None,
                "claim_amount": parsed.get("claim_amount_requested"),
                "has_narrative": bool(parsed.get("case_narrative")),
                "predicted_outcome": None,
                "predicted_probability": None,
                "all_probabilities": {},
                "parse_warnings": parsed.get("parse_warnings", ""),
                "skip_reason": skip_reason,
            })
            continue

        # Determine court_type from the court name
        court_type = features.infer_court_type(parsed.get("court") or "")

        # Build the prediction input from parsed fields
        case_args = dict(
            claim_amount=parsed.get("claim_amount_requested"),
            plaintiff_type=parsed.get("plaintiff_type"),
            defendant_type=parsed.get("defendant_type"),
            court_type=court_type,
            narrative_text=parsed.get("case_narrative"),
            judge_name=parsed.get("judge"),
            model=model,
        )

        ranked = predict_case(**case_args)
        top_outcome, top_prob = ranked[0]

        results.append({
            "source_file": filepath.name,
            "case_id": parsed.get("case_id"),
            "case_level": parsed.get("case_level"),
            "dispute_type": parsed.get("dispute_type"),
            "court": parsed.get("court"),
            "judge": parsed.get("judge"),
            "plaintiff_type": parsed.get("plaintiff_type"),
            "defendant_type": parsed.get("defendant_type"),
            "court_type": court_type,
            "claim_amount": parsed.get("claim_amount_requested"),
            "has_narrative": bool(parsed.get("case_narrative")),
            "predicted_outcome": top_outcome,
            "predicted_probability": round(top_prob * 100, 1),
            "all_probabilities": {o: round(p * 100, 1) for o, p in ranked},
            "parse_warnings": parsed.get("parse_warnings", ""),
            "skip_reason": None,
        })

    return results


def save_results_csv(results, output_path):
    """Write predictions to a CSV file."""
    if not results:
        return
    fieldnames = [
        "source_file", "case_id", "case_level", "dispute_type",
        "court", "judge", "plaintiff_type", "defendant_type",
        "court_type", "claim_amount", "has_narrative",
        "predicted_outcome", "predicted_probability",
        "all_probabilities", "parse_warnings", "skip_reason",
    ]
    with open(output_path, "w", newline="", encoding="utf-8-sig") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        w.writeheader()
        for row in results:
            row_copy = dict(row)
            # Flatten the probabilities dict to a readable string
            row_copy["all_probabilities"] = " | ".join(
                f"{o}: {p}%" for o, p in row["all_probabilities"].items()
            )
            w.writerow(row_copy)
    print(f"\nSaved → {output_path}")


if __name__ == "__main__":
    # In Jupyter, sys.argv might contain connection files like '-f /run/user/.../kernel.json'
    # We want to ignore those and use INPUT_FOLDER unless a real directory is passed.
    args = [arg for arg in sys.argv[1:] if not arg.startswith('-')]
    input_dir = args[0] if args else INPUT_FOLDER

    print(f"Scanning '{input_dir}/' for case files...\n")
    results = parse_and_predict_folder(input_dir)

    # Print per-file results
    for i, r in enumerate(results):
        print(f"{'='*60}")
        print(f"File:      {r['source_file']}")
        print(f"Case ID:   {r['case_id'] or '—'}")
        print(f"Court:     {r['court'] or '—'} ({r['court_type'] or '—'})")
        print(f"Judge:     {r['judge'] or '—'}")
        print(f"Parties:   {r['plaintiff_type'] or '?'} vs {r['defendant_type'] or '?'}")
        print(f"Claim ₮:   {r['claim_amount'] or '—'}")
        print(f"Narrative: {'yes' if r['has_narrative'] else 'NO (missing — prediction less reliable)'}")
        if r["parse_warnings"]:
            print(f"Warnings:  {r['parse_warnings']}")

        if r["skip_reason"]:
            print(f"\n⚠ SKIPPED - no prediction made: {r['skip_reason']}")
        else:
            print(f"\nPredicted outcome (ranked by probability):")
            for outcome, prob in r["all_probabilities"].items():
                marker = " ◀" if outcome == r["predicted_outcome"] else ""
                print(f"  {outcome:22s} {prob:5.1f}%{marker}")
        print()

    # Summary
    skipped = sum(1 for r in results if r["skip_reason"])
    print(f"{'='*60}")
    print(f"Total files processed: {len(results)}  ({len(results) - skipped} predicted, {skipped} skipped as out of scope)")

    # Save CSV
    save_results_csv(results, OUTPUT_CSV)

Scanning 'new_cases/' for case files...

File:      139_ШШ2020_00529.txt
Case ID:   —
Court:     — (—)
Judge:     —
Parties:   individual vs individual
Claim ₮:   —
Narrative: NO (missing — prediction less reliable)
Warnings:  meta_court_missing|meta_judge_missing|meta_case_index_missing|meta_case_id_missing|meta_decision_date_missing|meta_dispute_type_missing|case_level_no_court_field|outcome_no_point1|claim_amount_requested_not_found|law_articles_lead_in_not_found

⚠ SKIPPED - no prediction made: dispute_type could not be extracted - can't confirm this is even a loan dispute

File:      183_ШШ2020_01985.txt
Case ID:   —
Court:     хавтаст хэрэгт авагдсан бичгийн нотлох баримтуудыг нэг бүрчлэн шинжлэн судлав. (—)
Judge:     —
Parties:   individual vs company
Claim ₮:   —
Narrative: NO (missing — prediction less reliable)
Warnings:  meta_judge_missing|meta_case_index_missing|meta_case_id_missing|meta_decision_date_missing|meta_dispute_type_missing|outcome_no_point1|law_articles_lead_in

In [2]:
"""
Explanation-generation layer for the loan-dispute prediction pipeline.

Takes the structured extraction record + model prediction for a case,
builds an LLM prompt grounded ONLY in fields that were actually extracted
(nothing invented), and runs a cheap consistency check between the
generated explanation and the predicted label.

LLM-agnostic: plug in Claude / OpenAI / a local model by implementing
`llm_call(prompt: str) -> str` and passing it in.
"""

from dataclasses import dataclass, field
from typing import Callable, Dict, List, Optional


@dataclass
class CaseRecord:
    file: str
    case_id: Optional[str] = None
    court: Optional[str] = None
    judge: Optional[str] = None
    parties: Optional[str] = None
    claim_amount: Optional[str] = None
    narrative_present: bool = False
    dispute_type: Optional[str] = None
    law_articles: Optional[str] = None
    decision_date: Optional[str] = None
    warnings: List[str] = field(default_factory=list)


@dataclass
class Prediction:
    label: Optional[str] = None          # e.g. "claim_upheld" / "claim_rejected"
    confidence: Optional[float] = None
    skipped: bool = False
    skip_reason: Optional[str] = None


FIELD_LABELS: Dict[str, str] = {
    "court": "Court",
    "judge": "Judge",
    "parties": "Parties",
    "claim_amount": "Claim amount",
    "dispute_type": "Dispute type",
    "law_articles": "Law articles cited",
    "decision_date": "Decision date",
}


def _known_facts(case: CaseRecord) -> List[str]:
    """Only fields that were actually extracted -- nothing invented later."""
    facts = []
    for attr, label in FIELD_LABELS.items():
        value = getattr(case, attr)
        if value:
            facts.append(f"{label}: {value}")
    return facts


def build_explanation_prompt(case: CaseRecord, prediction: Prediction, language: str = "en") -> str:
    facts = _known_facts(case)
    facts_block = "\n".join(f"- {f}" for f in facts) if facts else "(no structured fields were reliably extracted)"

    reliability_note = ""
    if not case.narrative_present:
        reliability_note = (
            "Note: the case narrative text was not available, so base the explanation "
            "only on the fields below and say so explicitly.\n"
        )

    lang_name = "Mongolian" if language == "mn" else "English"

    return f"""You are writing a short explanation for a legal-outcome prediction, for a non-technical reader.

Predicted outcome: {prediction.label}
Confidence: {prediction.confidence}

Known facts about the case (use ONLY these -- do not invent amounts, articles, dates, or parties not listed):
{facts_block}

{reliability_note}Write 2-4 sentences in plain {lang_name}, referencing 2-3 of the facts above, explaining why the
model likely reached this prediction. Do not state anything not listed above. End with one sentence
naming the single weakest point in this evidence."""


def build_consistency_check_prompt(explanation: str, prediction: Prediction) -> str:
    return f"""Predicted label: {prediction.label}

Explanation:
\"\"\"{explanation}\"\"\"

Does the explanation's stated conclusion match the predicted label above?
Answer with exactly one word: MATCH or MISMATCH."""


def generate_explanation(
    case: CaseRecord,
    prediction: Prediction,
    llm_call: Callable[[str], str],
    language: str = "en",
) -> dict:
    if prediction.skipped:
        return {
            "explanation": None,
            "consistency": None,
            "status": "skipped",
            "reason": prediction.skip_reason,
        }

    prompt = build_explanation_prompt(case, prediction, language)
    explanation = llm_call(prompt).strip()

    check_prompt = build_consistency_check_prompt(explanation, prediction)
    check = llm_call(check_prompt).strip().upper()

    return {
        "explanation": explanation,
        "consistency": "match" if "MATCH" in check and "MISMATCH" not in check else "mismatch",
        "status": "ok",
        "reason": None,
    }


if __name__ == "__main__":
    # The exact skipped case from your pipeline output, as a smoke test.
    case = CaseRecord(
        file="139_\u0428\u04282020_00529.txt",
        parties="individual vs individual",
        narrative_present=False,
        warnings=[
            "meta_court_missing", "meta_judge_missing", "meta_case_index_missing",
            "meta_case_id_missing", "meta_decision_date_missing", "meta_dispute_type_missing",
            "case_level_no_court_field", "outcome_no_point1",
            "claim_amount_requested_not_found", "law_articles_lead_in_not_found",
        ],
    )
    prediction = Prediction(
        skipped=True,
        skip_reason="dispute_type could not be extracted - can't confirm this is even a loan dispute",
    )

    result = generate_explanation(case, prediction, llm_call=lambda p: "")
    print(result)
    # Expect: status "skipped", no LLM ever called -- matches the pipeline's own behavior.


{'explanation': None, 'consistency': None, 'status': 'skipped', 'reason': "dispute_type could not be extracted - can't confirm this is even a loan dispute"}


In [1]:
"""
Predict civil_outcome for real, new (undecided) first-instance loan cases
using the actual trained model.

Two different batch modes, for two different kinds of input files:

FULL-DOCUMENT MODE (default) - for testing against real scraped/decided
cases (files WITH a 'Шийдвэрийн мэдээлэл' metadata header, like anything
pulled from shuukh.mn):
    python3 predict_new_case.py [INPUT_FOLDER]

NARRATIVE-ONLY MODE - for genuinely new, undecided cases, where there IS
no metadata header yet because nothing's been decided - just the
plaintiff's petition text, one case per .txt file:
    python3 predict_new_case.py --narrative-only [INPUT_FOLDER] [--court-type urban|rural]

    Everything except the narrative gets a sensible default (claim
    amount and judge caseload fall back to training medians, party
    types to 'unknown') - the narrative itself carries most of the
    model's predictive signal anyway, so this is still meaningful, just
    somewhat less precise than providing the real structured facts too.

Both modes save a summary CSV (predictions_output.csv).

SINGLE-CASE API (most accurate - use this directly when you know the
real facts, rather than either batch mode):
    from predict_new_case import predict_case
    results = predict_case(claim_amount=..., plaintiff_type=..., ...)

Needs: trained_model.joblib in the same folder (run train_and_save_model.py
first if it doesn't exist yet).
"""
import csv
import sys
from pathlib import Path

import joblib
from rich.console import Console
from rich.panel import Panel
from rich.table import Table
from rich.text import Text

import features
import shuukh_parser_v2 as parser

MODEL_PATH = "trained_model.joblib"
INPUT_FOLDER = "new_cases"
OUTPUT_CSV = "predictions_output.csv"

console = Console()

OUTCOME_COLORS = {
    "granted": "green",
    "granted_uncontested": "bright_green",
    "settled": "cyan",
    "partial": "yellow",
    "denied": "red",
    "withdrawn": "grey58",
}
OUTCOME_LABELS = {
    "granted": "GRANTED",
    "granted_uncontested": "GRANTED (uncontested)",
    "settled": "SETTLED",
    "partial": "PARTIALLY GRANTED",
    "denied": "DENIED",
    "withdrawn": "WITHDRAWN",
}


def confidence_label(ranked):
    """ranked: [(outcome, probability_0_to_1), ...] sorted highest first.
    Requires BOTH a clear margin over the runner-up AND a reasonably high
    absolute probability - margin alone isn't enough. A 43% leader with a
    big gap to 2nd place still means the model thinks there's a >50%
    chance the real outcome is something else; calling that "high
    confidence" would overstate what the model actually knows."""
    top_p = ranked[0][1]
    second_p = ranked[1][1] if len(ranked) > 1 else 0
    margin = top_p - second_p
    if margin >= 0.20 and top_p >= 0.50:
        return "HIGH CONFIDENCE", "green"
    elif margin >= 0.08 or top_p >= 0.30:
        return "MODERATE CONFIDENCE", "yellow"
    return "LOW CONFIDENCE — close call", "red"


def load_model():
    saved = joblib.load(MODEL_PATH)
    return saved["pipeline"], saved["median_claim"], saved["judge_counts"]


def predict_case(claim_amount, plaintiff_type, defendant_type, court_type,
                  narrative_text, judge_name=None, model=None):
    """
    claim_amount: number (₮), or None if unknown
    plaintiff_type / defendant_type: 'individual' or 'company'
    court_type: 'urban' (Ulaanbaatar district court) or 'rural' (aimag/province)
    narrative_text: the actual petition/claim text, in Mongolian - this is
        the single most informative input (TF-IDF over this text is now
        the model's top feature group), don't skip it if it's available
    judge_name: pass this if the judge is already assigned and known -
        otherwise the model uses the median caseload across training data
    """
    if model is None:
        pipe, median_claim, judge_counts = load_model()
    else:
        pipe, median_claim, judge_counts = model

    row = features.build_case_features(
        claim_amount=claim_amount,
        plaintiff_type=plaintiff_type,
        defendant_type=defendant_type,
        court_type=court_type,
        narrative_text=narrative_text,
        judge_name=judge_name,
        median_claim=median_claim,
        judge_counts=judge_counts,
    )
    proba = pipe.predict_proba(row)[0]
    classes = pipe.classes_
    ranked = sorted(zip(classes, proba), key=lambda x: -x[1])
    return ranked  # [(outcome, probability), ...] sorted highest first


def parse_and_predict_folder(input_folder, model=None):
    """
    Scan input_folder for .txt files, parse each with shuukh_parser_v2,
    extract the features the trained model needs, and return predictions.

    Returns a list of dicts, one per file, with parsed metadata + predictions.
    """
    folder = Path(input_folder)
    if not folder.is_dir():
        print(f"ERROR: Input folder '{folder}' does not exist.")
        print(f"Create it and drop .txt case files there, then re-run.")
        sys.exit(1)

    txt_files = sorted(
        p for p in folder.glob("*")
        if p.is_file() and not p.name.startswith(".") and p.name != "README.md" and p.name != OUTPUT_CSV
    )
    if not txt_files:
        print(f"No valid case files found in '{folder}/'.")
        print(f"Drop case narrative files there and re-run.")
        sys.exit(0)

    if model is None:
        model = load_model()

    results = []
    for filepath in txt_files:
        parsed = parser.parse_file(filepath)
        dispute_type = parsed.get("dispute_type") or ""
        case_level = parsed.get("case_level")

        # Refuse to predict on anything we can't confirm is in scope,
        # rather than silently giving a confident-looking prediction on
        # something the model was never trained on. This model was only
        # trained on first_instance "Бусад зээл" cases specifically -
        # note that's non-bank lending (ББСБ, app-based, installment,
        # gov-fund, individual loans) - "Банкны зээл" (bank loans) is a
        # SEPARATE category on the source site and was never in scope.
        skip_reason = None
        if not dispute_type:
            skip_reason = "dispute_type could not be extracted - can't confirm this is even a loan dispute"
        elif "Бусад зээл" not in dispute_type:
            skip_reason = f"dispute_type is '{dispute_type}', not 'Бусад зээл' - out of scope for this model (e.g. a bank loan, lease, or other dispute type wasn't part of training)"
        elif case_level != "first_instance":
            skip_reason = f"case_level is '{case_level}', not 'first_instance' - this model only predicts first-instance outcomes"

        if skip_reason:
            results.append({
                "source_file": filepath.name,
                "case_id": parsed.get("case_id"),
                "case_level": case_level,
                "dispute_type": parsed.get("dispute_type"),
                "court": parsed.get("court"),
                "judge": parsed.get("judge"),
                "plaintiff_type": parsed.get("plaintiff_type"),
                "defendant_type": parsed.get("defendant_type"),
                "court_type": None,
                "claim_amount": parsed.get("claim_amount_requested"),
                "has_narrative": bool(parsed.get("case_narrative")),
                "predicted_outcome": None,
                "predicted_probability": None,
                "all_probabilities": {},
                "parse_warnings": parsed.get("parse_warnings", ""),
                "skip_reason": skip_reason,
            })
            continue

        # Determine court_type from the court name
        court_type = features.infer_court_type(parsed.get("court") or "")

        # Build the prediction input from parsed fields
        case_args = dict(
            claim_amount=parsed.get("claim_amount_requested"),
            plaintiff_type=parsed.get("plaintiff_type"),
            defendant_type=parsed.get("defendant_type"),
            court_type=court_type,
            narrative_text=parsed.get("case_narrative"),
            judge_name=parsed.get("judge"),
            model=model,
        )

        ranked = predict_case(**case_args)
        top_outcome, top_prob = ranked[0]

        results.append({
            "source_file": filepath.name,
            "case_id": parsed.get("case_id"),
            "case_level": parsed.get("case_level"),
            "dispute_type": parsed.get("dispute_type"),
            "court": parsed.get("court"),
            "judge": parsed.get("judge"),
            "plaintiff_type": parsed.get("plaintiff_type"),
            "defendant_type": parsed.get("defendant_type"),
            "court_type": court_type,
            "claim_amount": parsed.get("claim_amount_requested"),
            "has_narrative": bool(parsed.get("case_narrative")),
            "predicted_outcome": top_outcome,
            "predicted_probability": round(top_prob * 100, 1),
            "all_probabilities": {o: round(p * 100, 1) for o, p in ranked},
            "parse_warnings": parsed.get("parse_warnings", ""),
            "skip_reason": None,
        })

    return results


def predict_narrative_only_folder(input_folder, model=None, court_type="urban"):
    """
    For files that are JUST the plaintiff's narrative text - no
    'Шийдвэрийн мэдээлэл' header, no case metadata. This is what a
    genuinely new, undecided case's petition actually looks like: there's
    no metadata to have yet, nothing's been decided. Each ENTIRE .txt
    file is treated as one case's narrative, full stop.

    Everything except the narrative uses a default: claim_amount and
    judge_case_count fall back to the training medians, party types to
    'unknown'. court_type defaults to 'urban' since it's not something
    narrative text alone tells you - pass the real value if you know it,
    since it does meaningfully affect the prediction.
    """
    folder = Path(input_folder)
    if not folder.is_dir():
        print(f"ERROR: Input folder '{folder}' does not exist.")
        print(f"Create it and drop plain narrative .txt files there, then re-run.")
        sys.exit(1)

    txt_files = sorted(p for p in folder.glob("*.txt") if p.is_file())
    if not txt_files:
        print(f"No .txt files found in '{folder}/'.")
        sys.exit(0)

    if model is None:
        model = load_model()

    results = []
    for filepath in txt_files:
        narrative_text = filepath.read_text(encoding="utf-8", errors="replace").strip()
        ranked = predict_case(
            claim_amount=None,
            plaintiff_type=None,
            defendant_type=None,
            court_type=court_type,
            narrative_text=narrative_text,
            judge_name=None,
            model=model,
        )
        top_outcome, top_prob = ranked[0]
        results.append({
            "source_file": filepath.name,
            "case_id": None,
            "case_level": None,
            "dispute_type": None,
            "court": None,
            "judge": None,
            "plaintiff_type": None,
            "defendant_type": None,
            "court_type": court_type,
            "claim_amount": None,
            "has_narrative": bool(narrative_text),
            "predicted_outcome": top_outcome,
            "predicted_probability": round(top_prob * 100, 1),
            "all_probabilities": {o: round(p * 100, 1) for o, p in ranked},
            "parse_warnings": "",
            "skip_reason": None,
        })

    return results


def display_result(r):
    """One rich panel per case - colored probability bars, a confidence
    read, bordered in the predicted outcome's color so a quick glance
    down a long batch run tells you a lot before reading any text."""
    if r.get("skip_reason"):
        console.print(Panel(
            f"[bold red]No prediction made[/bold red]\n{r['skip_reason']}",
            title=f"[dim]{r['source_file']}[/dim]",
            border_style="red",
        ))
        return

    ranked = sorted(r["all_probabilities"].items(), key=lambda x: -x[1])
    top_outcome = r["predicted_outcome"]
    conf_label, conf_color = confidence_label([(o, p / 100) for o, p in ranked])

    facts = f"{r.get('plaintiff_type') or '?'} vs {r.get('defendant_type') or '?'}"
    if r.get("claim_amount"):
        facts += f"  ·  ₮{r['claim_amount']:,.0f}"
    if r.get("court_type"):
        facts += f"  ·  {'Ulaanbaatar' if r['court_type'] == 'urban' else 'Aimag'} court"
    if not r.get("has_narrative"):
        facts += "  ·  [red]no narrative — less reliable[/red]"

    body = Text.from_markup(f"[dim]{facts}[/dim]\n\n")
    for outcome, p in ranked:
        color = OUTCOME_COLORS.get(outcome, "white")
        label = OUTCOME_LABELS.get(outcome, outcome)
        bar_len = round(p / 100 * 28)
        bar = "█" * bar_len + "░" * (28 - bar_len)
        is_top = outcome == top_outcome
        style = f"bold {color}" if is_top else "dim"
        marker = "  ◀" if is_top else ""
        body.append(f"{label:24s} ", style=style)
        body.append(f"{bar} ", style=color if is_top else "dim " + color)
        body.append(f"{p:5.1f}%{marker}\n", style=style)
    body.append(f"\n{conf_label}", style=f"bold {conf_color}")

    console.print(Panel(
        body,
        title=f"[bold]{r['source_file']}[/bold]",
        subtitle=f"[bold {OUTCOME_COLORS.get(top_outcome, 'white')}]{OUTCOME_LABELS.get(top_outcome, top_outcome)}[/]",
        border_style=OUTCOME_COLORS.get(top_outcome, "white"),
    ))


def display_summary(results):
    predicted = [r for r in results if not r.get("skip_reason")]
    skipped = [r for r in results if r.get("skip_reason")]

    table = Table(title=f"{len(results)} files processed", show_lines=False)
    table.add_column("Outcome")
    table.add_column("Count", justify="right")
    from collections import Counter
    counts = Counter(r["predicted_outcome"] for r in predicted)
    for outcome, label in OUTCOME_LABELS.items():
        if counts[outcome]:
            table.add_row(f"[{OUTCOME_COLORS[outcome]}]{label}[/{OUTCOME_COLORS[outcome]}]", str(counts[outcome]))
    if skipped:
        table.add_row("[red]Skipped (out of scope)[/red]", str(len(skipped)))
    console.print(table)


def save_results_csv(results, output_path):
    """Write predictions to a CSV file."""
    if not results:
        return
    fieldnames = [
        "source_file", "case_id", "case_level", "dispute_type",
        "court", "judge", "plaintiff_type", "defendant_type",
        "court_type", "claim_amount", "has_narrative",
        "predicted_outcome", "predicted_probability",
        "all_probabilities", "parse_warnings", "skip_reason",
    ]
    with open(output_path, "w", newline="", encoding="utf-8-sig") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        w.writeheader()
        for row in results:
            row_copy = dict(row)
            # Flatten the probabilities dict to a readable string
            row_copy["all_probabilities"] = " | ".join(
                f"{o}: {p}%" for o, p in row["all_probabilities"].items()
            )
            w.writerow(row_copy)
    print(f"\nSaved → {output_path}")


if __name__ == "__main__":
    # In Jupyter, sys.argv might contain connection files like '-f /run/user/.../kernel.json'
    # We want to ignore those, plus pull out our own flags before treating
    # whatever's left as the folder path.
    raw_args = sys.argv[1:]
    narrative_only = "--narrative-only" in raw_args
    court_type_override = "rural" if "--court-type" in raw_args and "rural" in raw_args else None

    positional = [
        a for a in raw_args
        if not a.startswith("-") and a not in ("urban", "rural")
    ]
    input_dir = positional[0] if positional else INPUT_FOLDER

    if narrative_only:
        court_type = court_type_override or "urban"
        console.print(f"[dim]Scanning '{input_dir}/' for narrative-only case files (court_type={court_type})...[/dim]\n")
        results = predict_narrative_only_folder(input_dir, court_type=court_type)
    else:
        console.print(f"[dim]Scanning '{input_dir}/' for case files...[/dim]\n")
        results = parse_and_predict_folder(input_dir)

    for r in results:
        display_result(r)
        console.print()

    display_summary(results)

    # Save CSV
    save_results_csv(results, OUTPUT_CSV)

Scanning 'new_cases/' for case files...

╭───────────────────────────────────────────── 139_ШШ2020_00529.txt ──────────────────────────────────────────────╮
│ No prediction made                                                                                              │
│ dispute_type could not be extracted - can't confirm this is even a loan dispute                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── 183_ШШ2020_01985.txt ──────────────────────────────────────────────╮
│ No prediction made                                                                                              │
│ dispute_type could not be extracted - can't confirm this is even a loan dispute                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── 310_ШШ2026_00544.txt ──────────────────────────────────────────────╮
│ No prediction made                                                                                              │
│ dispute_type could not be extracted - can't confirm this is even a loan dispute                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

        3 files processed         
┏━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━┓
┃ Outcome                ┃ Count ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━┩
│ Skipped (out of scope) │     3 │
└────────────────────────┴───────┘


Saved → predictions_output.csv
